In [22]:
#Initial Imports
from astropy.io import fits
from scipy.ndimage import median_filter
import numpy as np
from photutils.detection import DAOStarFinder
from scipy.optimize import curve_fit, linear_sum_assignment
import matplotlib.pyplot as plt
import math
import pandas as pd
import re
import matplotlib.colors as mcolors
import colorsys
import os

In [ ]:
#list of fits file paths paired with detector temperatures at the time of observation
#stored as a list of tuples (filepath, temp(float))
temp_pairs = [(r"Y:\2D\20260406\iLocater_lab_20260406_0001_hxrgproc.fits", 130.738),
(r"Y:\2D\20260406\iLocater_lab_20260406_0002_hxrgproc.fits", 130.755),
(r"Y:\2D\20260407\iLocater_lab_20260407_0001_hxrgproc.fits", 121.429),
(r"Y:\2D\20260407\iLocater_lab_20260407_0002_hxrgproc.fits", 121.461),
(r"Y:\2D\20260409\iLocater_lab_20260409_0001_hxrgproc.fits", 102.087),
(r"Y:\2D\20260409\iLocater_lab_20260409_0002_hxrgproc.fits", 102.078),
(r"Y:\2D\20260410\iLocater_lab_20260410_0001_hxrgproc.fits", 95.8237),
(r"Y:\2D\20260410\iLocater_lab_20260410_0002_hxrgproc.fits", 95.8189),
(r"Y:\2D\20260411\iLocater_lab_20260411_0001_hxrgproc.fits", 90.8267),
(r"Y:\2D\20260411\iLocater_lab_20260411_0002_hxrgproc.fits", 90.8215),
(r"Y:\2D\20260412\iLocater_lab_20260412_0001_hxrgproc.fits", 88.1519),
(r"Y:\2D\20260412\iLocater_lab_20260412_0002_hxrgproc.fits", 88.1490),
(r"Y:\2D\20260413\iLocater_lab_20260413_0001_hxrgproc.fits", 86.4587),
(r"Y:\2D\20260413\iLocater_lab_20260413_0002_hxrgproc.fits", 86.4556),
(r"Y:\2D\20260414\iLocater_lab_20260414_0001_hxrgproc.fits", 84.8636),
(r"Y:\2D\20260414\iLocater_lab_20260414_0002_hxrgproc.fits", 84.8621),
(r"Y:\2D\20260415\iLocater_lab_20260415_0001_hxrgproc.fits", 83.7054),
(r"Y:\2D\20260415\iLocater_lab_20260415_0002_hxrgproc.fits", 83.7017),
(r"Y:\2D\20260416\iLocater_lab_20260416_0016_hxrgproc.fits", 84.1627),
(r"Y:\2D\20260416\iLocater_lab_20260416_0017_hxrgproc.fits", 84.1642),
(r"Y:\2D\20260417\iLocater_lab_20260417_0001_hxrgproc.fits", 84.41245),
(r"Y:\2D\20260417\iLocater_lab_20260417_0002_hxrgproc.fits", 84.4123),
(r"Y:\2D\20260420\iLocater_lab_20260420_0003_hxrgproc.fits", 84.69485),
(r"Y:\2D\20260420\iLocater_lab_20260420_0004_hxrgproc.fits", 84.6964),
(r"Y:\2D\20260421\iLocater_lab_20260421_0012_hxrgproc.fits", 84.69515),
(r"Y:\2D\20260421\iLocater_lab_20260421_0013_hxrgproc.fits", 84.6944),
(r"Y:\2D\20260422\iLocater_lab_20260422_0001_hxrgproc.fits", 84.6951),
(r"Y:\2D\20260422\iLocater_lab_20260422_0002_hxrgproc.fits", 84.69465),
(r"Y:\2D\20260423\iLocater_lab_20260423_0016_hxrgproc.fits", 84.6940),
(r"Y:\2D\20260423\iLocater_lab_20260423_0017_hxrgproc.fits", 84.6929),
(r"Y:\2D\20260424\iLocater_lab_20260424_0001_hxrgproc.fits", 84.6954),
(r"Y:\2D\20260424\iLocater_lab_20260424_0002_hxrgproc.fits", 84.6454)]

output_csv = r"Dark_Current_Summary_Stats.csv"


In [ ]:
"""
Fits file processing function. 
Takes lists of filepaths and their respective detector temperatures.
Calculates mean current across the entire image, as well as standard deivation and variance.
Generates a scatterplot of detector temp vs mean current, and saves all stats in a CSV called "Dark_Current_Summary_Stats.csv"
CSV saving is optional and can be turned off by changing the "exportcsv" kwarg to 0.
"""
def process_fits(temp_pairs = temp_pairs, exportcsv=1):
    rows = []
    for path, temp in temp_pairs:
        print(path)
        data = fits.getdata(path)

        n = data.size
        total = np.sum(data)
        mean = np.nanmean(data)
        std = np.nanstd(data)
        sem = std / np.sqrt(n)
        var = std**2

        #build a list of stats for the files in the input list
        rows.append({
            "Filename": os.path.basename(path),
            "Temp": temp,
            "Total Readout": total,
            "Total Readout Uncertainty": n * sem,
            "Mean Readout (Per Pixel)": mean,
            "Standard Deviation (Per Pixel)": std,
            "Standard Error in the Pixel Mean": sem,
            "Variance (Per Pixel)": var
        })
    
    #plot data vs temperature
    df = pd.DataFrame(rows)
    t = df["Temp"].to_numpy()
    y2 = df["Mean Readout (Per Pixel)"].to_numpy()
    y1 = df["Total Readout"].to_numpy()
    y2err = df["Standard Error in the Pixel Mean"].to_numpy()
    y1err = df["Total Readout Uncertainty"].to_numpy()

    even = np.arange(len(t)) % 2 == 0
    odd = ~even

    #plot the total dark current per image
    plt.figure()
    plt.suptitle("Dark Current Stats by Detector Temperature")
    
    #plot per pixel stats
    plt.scatter(
        t[even], y2[even],
        marker="^", c="red", alpha=0.75, label="20-Frame Images"
    )
    
    plt.scatter(
        t[odd], y2[odd], 
        marker="v", c="green", alpha=0.75, label="150-Frame Images"
    )
    plt.ylabel("Mean Slope Per Pixel")
    plt.ylim(-0.1, 0.1)
    plt.xlabel("Detector Temperature (K)")
    plt.xlim(80, 105)
    plt.grid(True, alpha=0.6, which="both")
    plt.legend()

    plt.show()

    #turn the data list into an exported csv
    if exportcsv == True:
        df.to_csv(output_csv, index=False)

In [ ]:
#Run fits processing function
process_fits()

Y:\2D\20260406\iLocater_lab_20260406_0001_hxrgproc.fits
Y:\2D\20260406\iLocater_lab_20260406_0002_hxrgproc.fits
Y:\2D\20260407\iLocater_lab_20260407_0001_hxrgproc.fits
Y:\2D\20260407\iLocater_lab_20260407_0002_hxrgproc.fits
Y:\2D\20260409\iLocater_lab_20260409_0001_hxrgproc.fits
Y:\2D\20260409\iLocater_lab_20260409_0002_hxrgproc.fits
Y:\2D\20260410\iLocater_lab_20260410_0001_hxrgproc.fits
Y:\2D\20260410\iLocater_lab_20260410_0002_hxrgproc.fits
Y:\2D\20260411\iLocater_lab_20260411_0001_hxrgproc.fits
Y:\2D\20260411\iLocater_lab_20260411_0002_hxrgproc.fits
Y:\2D\20260412\iLocater_lab_20260412_0001_hxrgproc.fits
Y:\2D\20260412\iLocater_lab_20260412_0002_hxrgproc.fits
Y:\2D\20260413\iLocater_lab_20260413_0001_hxrgproc.fits
Y:\2D\20260413\iLocater_lab_20260413_0002_hxrgproc.fits
Y:\2D\20260414\iLocater_lab_20260414_0001_hxrgproc.fits
Y:\2D\20260414\iLocater_lab_20260414_0002_hxrgproc.fits
Y:\2D\20260415\iLocater_lab_20260415_0001_hxrgproc.fits
Y:\2D\20260415\iLocater_lab_20260415_0002_hxrgpr